# Load signed quotes and make them look like normal trades

In [ ]:
import pandas as pd

df = pd.read_parquet("/Volumes/Extreme SSD/data/trades_and_quotes/20250516.parquet")

In [ ]:
df['tick_type'].unique()

In [ ]:
import numpy as np
df['is_trade'] = False
df.loc[df['price'].notnull(), 'is_trade'] = True

In [ ]:
df['is_trade'].value_counts()/len(df)

In [ ]:
df.loc['bsp_quote_imb'] = 0
df.loc[df['tick_type']=='ask_size_up', 'bsp_quote_imb'] = -1
df.loc[df['tick_type']=='bid_size_up', 'bsp_quote_imb'] = 1

In [ ]:
frame = df[(df['is_trade']==False)&(df['bsp_quote_imb'].abs()>0)].copy()

In [ ]:
frame['bsp'] = frame['bsp_quote_imb'].copy()

In [ ]:
len(frame)

In [ ]:
len(frame['ticker'].unique())

In [ ]:
len(df[(df['is_trade']==False)])

In [ ]:
df[(df['is_trade']==False)]['tick_type'].value_counts()

In [ ]:
import pandas as pd
import kernel_g
import filters


start_time="10:00:06"
end_time="15:00:50"
shuffle='decisecond'
shift_ns=10_000_000
grid_ns=100_000_000
n_periods=50
upper_count=3


In [ ]:
import kernel_g
import util
import polars as pl

In [ ]:
d = frame.copy()

In [ ]:
d = pl.from_pandas(d)

In [ ]:
n_shifts = grid_ns // shift_ns
assert grid_ns % shift_ns == 0, f"grid_ns={grid_ns} should be divisible by shift_ns={shift_ns}"

results = []
for i in range(0, n_shifts):
    # prepare frame doesn't modify inputs
    df_local = kernel_g.prepare_frame_general(df=d, i=i, shift_ns=shift_ns, grid_ns=grid_ns, n_periods=n_periods)
    r = kernel_g.get_z_score_general(d=df_local, upper_count=upper_count, n_periods=n_periods, quantile_clip=False)
    r = r.with_columns(pl.lit(i).alias('shift'))
    r = r.with_columns(
        pl.concat_str([
            pl.col('grid_mod_n'),
            pl.col('shift')
        ], separator='_').alias('kernel')
    )
    results.append(r)
    # statistic = results_to_t(r)
    # results.append(statistic)

not_random_frame = pl.concat(results)


    # random version
d_r = d.clone()

random_results = []
for i in range(0, 10):
    # prepare frame doesn't modify the input
    df_local = d_r.clone()
    # we do shuffling after defining a new grid but both options are reasonable
    if shuffle=='second':
        df_local = util.add_block_shuffled_time(df_local, block=1000_000_000)
    elif shuffle=='decisecond':
        df_local = util.add_block_shuffled_time(df_local)
    else:
        raise NotImplementedError("the value should be decisecond or second")

    # define new grid
    # preparing the frame MUST BE AFTER SHUFFLING, NOT BEFORE
    df_local = kernel_g.prepare_frame_general(df=df_local, i=i, shift_ns=shift_ns, grid_ns=grid_ns, n_periods=n_periods)

    r = kernel_g.get_z_score_general(d=df_local, upper_count=upper_count, n_periods=n_periods, quantile_clip=False)
    r = r.with_columns(pl.lit(i).alias('shift'))
    r = r.with_columns(
        pl.concat_str([
            pl.col('grid_mod_n'),
            pl.col('shift')
        ], separator='_').alias('kernel')
    )

    random_results.append(r)
    # statistic = results_to_t(r)
    # random_results.append(statistic)

random_frame = pl.concat(random_results)

In [ ]:
f = kernel_g.kernel_stats_pl(not_random_frame)
f = f.to_pandas()

r = kernel_g.kernel_stats_pl(random_frame)
r = r.to_pandas()

f['type'] = "NOT_RANDOM"
r['type'] = "RANDOM"

f = f.drop_duplicates(subset='kernel')
r = r.drop_duplicates(subset='kernel')

d = pd.concat([f, r])

In [ ]:
import plotly.express as px

px.histogram(d, color='type', x='top_5_z_score_abs')

In [ ]:
col = "top_5_z_score_abs"
n = f.set_index(['grid_mod_n', 'shift']).sort_index()[col].unstack(level=0)

lower = r[col].quantile(0.99)
upper = r[col].max() + 0.5
n = n.clip(upper=upper, lower=lower)

px.imshow(n)